In [1]:
# 1) Install PySpark

!pip -q install pyspark

import os, time, glob, shutil
from datetime import datetime

from google.colab import drive
drive.mount("/content/drive")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler,
    Imputer, StandardScaler
)
from pyspark.ml.classification import (
    LogisticRegression, DecisionTreeClassifier,
    RandomForestClassifier, GBTClassifier
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 2) Spark Session (Colab memory guard)
spark = (
    SparkSession.builder
    .appName("7006SCN_CoAt_PySpark_Colab")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [3]:
# 3) EDIT THESE PATHS
PARQUET_PATH = "/content/drive/MyDrive/7006SCN/data/CoAt_NF-UQ-NIDS-V2.parquet"

# Separate output directories (different folders)
BASE_OUT = "/content/drive/MyDrive/7006SCN_Out/outputs"
SAMPLE_DIR_BASE  = os.path.join(BASE_OUT, "sample_data")
METRICS_DIR_BASE = os.path.join(BASE_OUT, "model_metrics")

os.makedirs(SAMPLE_DIR_BASE, exist_ok=True)
os.makedirs(METRICS_DIR_BASE, exist_ok=True)

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
SAMPLE_OUT_DIR = os.path.join(SAMPLE_DIR_BASE,  f"sample_120k_{RUN_TAG}")     # spark folder
METRICS_OUT_DIR = os.path.join(METRICS_DIR_BASE, f"metrics_{RUN_TAG}")        # spark folder

SAMPLE_SINGLE_CSV  = os.path.join(SAMPLE_DIR_BASE,  f"sample_120k_{RUN_TAG}.csv")
METRICS_SINGLE_CSV = os.path.join(METRICS_DIR_BASE, f"model_metrics_{RUN_TAG}.csv")


In [4]:
# 4) Helper: detect label column
def detect_label_col(df):
    candidates = ["label", "Label", "LABEL", "class", "Class", "target", "Target", "y", "attack", "Attack"]
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        if "label" in cl or "class" in cl or "target" in cl:
            return c
    return None

In [5]:
# 5) Load Parquet (LAZY) - no count() here
df = spark.read.parquet(PARQUET_PATH)

label_col = detect_label_col(df)
if label_col is None:
    raise ValueError(
        "Could not detect label column. Rename target to label/Label/class/target/attack OR edit detect_label_col()."
    )
print("Detected label column:", label_col)

Detected label column: Label


In [6]:
# 6) Create binary label column "label_bin"
label_dtype = dict(df.dtypes)[label_col]

if label_dtype in ("int", "bigint", "double", "float", "smallint", "tinyint"):
    df = df.withColumn(
        "label_bin",
        F.when(F.col(label_col).cast("double") > F.lit(0.0), F.lit(1.0)).otherwise(F.lit(0.0))
    )
else:
    benign_set = ["benign", "normal", "legitimate", "legit", "background"]
    df = df.withColumn("label_str", F.lower(F.col(label_col).cast("string")))
    df = df.withColumn(
        "label_bin",
        F.when(F.col("label_str").isin(benign_set), F.lit(0.0)).otherwise(F.lit(1.0))
    ).drop("label_str")

df = df.withColumn("label_bin", F.col("label_bin").cast("double"))

In [7]:
# 7) Build feature list (avoid empty-name columns)
drop_cols = set([label_col, "label_bin"])
for c in df.columns:
    cl = c.lower()
    if cl in ("id", "flow_id", "timestamp", "time", "date") or "uuid" in cl:
        drop_cols.add(c)

feature_cols = [c for c in df.columns if c not in drop_cols and isinstance(c, str) and c.strip() != ""]
if not feature_cols:
    raise ValueError("No feature columns found after dropping label/id columns. Review drop_cols logic.")

df_feat = df.select(*(feature_cols + ["label_bin"]))

In [8]:
# 8) Materialize Data

SAMPLE_N = 120_000

USE_PROGRESSIVE_SAMPLING = True  # set False to use just .limit()

if not USE_PROGRESSIVE_SAMPLING:
    # Fastest (but takes first rows -> may be biased)
    df_sample = df_feat.limit(SAMPLE_N)
else:
    # Progressive sampling: reads only what needed; counts only the SMALL sample, not full dataset
    frac = 0.001
    df_sample = df_feat.sample(withReplacement=False, fraction=frac, seed=42).limit(SAMPLE_N)
    got = df_sample.count()  # count on sampled subset only (small)
    while got < SAMPLE_N and frac < 1.0:
        frac = min(frac * 2, 1.0)
        df_sample = df_feat.sample(withReplacement=False, fraction=frac, seed=42).limit(SAMPLE_N)
        got = df_sample.count()
    if got < SAMPLE_N:
        # fallback (still not full scan count)
        df_sample = df_feat.limit(SAMPLE_N)

print("Sample ready.")

Sample ready.


In [9]:
# 9) Save sample as CSV in SAMPLE directory (folder + single file)
df_sample.coalesce(1).write.mode("overwrite").option("header", True).csv(SAMPLE_OUT_DIR)

part_files = glob.glob(os.path.join(SAMPLE_OUT_DIR, "part-*.csv"))
if not part_files:
    raise RuntimeError("Could not find sample part-*.csv in: " + SAMPLE_OUT_DIR)
shutil.copy(part_files[0], SAMPLE_SINGLE_CSV)

print("Sample folder:", SAMPLE_OUT_DIR)
print("Sample single CSV:", SAMPLE_SINGLE_CSV)


Sample folder: /content/drive/MyDrive/7006SCN_Out/outputs/sample_data/sample_120k_20260227_082713
Sample single CSV: /content/drive/MyDrive/7006SCN_Out/outputs/sample_data/sample_120k_20260227_082713.csv


In [10]:
# 10) Split train/test (only on sample)
train_df, test_df = df_sample.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.persist()
test_df  = test_df.persist()

In [11]:
# 11) Auto-build preprocessing pipeline
schema = df_sample.schema

numeric_cols, categorical_cols = [], []
for c in feature_cols:
    dt = schema[c].dataType
    if isinstance(dt, (T.IntegerType, T.LongType, T.DoubleType, T.FloatType, T.ShortType, T.ByteType, T.DecimalType)):
        numeric_cols.append(c)
    else:
        categorical_cols.append(c)

stages = []

# Impute numeric columns
if numeric_cols:
    imputer = Imputer(inputCols=numeric_cols, outputCols=[f"{c}__imputed" for c in numeric_cols])
    stages.append(imputer)
    numeric_out = [f"{c}__imputed" for c in numeric_cols]
else:
    numeric_out = []

# Encode categorical columns
ohe_cols = []
for c in categorical_cols:
    idx = StringIndexer(inputCol=c, outputCol=f"{c}__idx", handleInvalid="keep")
    ohe = OneHotEncoder(inputCols=[f"{c}__idx"], outputCols=[f"{c}__ohe"], handleInvalid="keep")
    stages.extend([idx, ohe])
    ohe_cols.append(f"{c}__ohe")

assembler_inputs = [x for x in (numeric_out + ohe_cols) if x and x.strip() != ""]
if not assembler_inputs:
    raise ValueError("Assembler has no inputs. Check feature columns.")

assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep")
stages.append(assembler)

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withMean=False, withStd=True)
stages.append(scaler)


In [12]:
# 12) Four MLlib models
models = [
    ("LogisticRegression", LogisticRegression(featuresCol="scaledFeatures", labelCol="label_bin", maxIter=50)),
    ("DecisionTree", DecisionTreeClassifier(featuresCol="scaledFeatures", labelCol="label_bin", maxDepth=10)),
    ("RandomForest", RandomForestClassifier(featuresCol="scaledFeatures", labelCol="label_bin", numTrees=100, maxDepth=12)),
    ("GBT", GBTClassifier(featuresCol="scaledFeatures", labelCol="label_bin", maxIter=50, maxDepth=6)),
]

# Evaluators
auc_eval  = BinaryClassificationEvaluator(labelCol="label_bin", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
acc_eval  = MulticlassClassificationEvaluator(labelCol="label_bin", predictionCol="prediction", metricName="accuracy")
f1_eval   = MulticlassClassificationEvaluator(labelCol="label_bin", predictionCol="prediction", metricName="f1")
prec_eval = MulticlassClassificationEvaluator(labelCol="label_bin", predictionCol="prediction", metricName="weightedPrecision")
rec_eval  = MulticlassClassificationEvaluator(labelCol="label_bin", predictionCol="prediction", metricName="weightedRecall")


In [13]:
# 13) Train + Evaluate
results = []
for name, clf in models:
    print("\n==============================")
    print("Training:", name)
    print("==============================")

    pipeline = Pipeline(stages=stages + [clf])

    start = time.time()
    fitted = pipeline.fit(train_df)
    train_time = time.time() - start

    pred = fitted.transform(test_df)

    auc = float(auc_eval.evaluate(pred))
    acc = float(acc_eval.evaluate(pred))
    f1  = float(f1_eval.evaluate(pred))
    wp  = float(prec_eval.evaluate(pred))
    wr  = float(rec_eval.evaluate(pred))

    results.append({
        "model": name,
        "train_time_sec": round(train_time, 4),
        "AUC_ROC": round(auc, 6),
        "accuracy": round(acc, 6),
        "f1": round(f1, 6),
        "weightedPrecision": round(wp, 6),
        "weightedRecall": round(wr, 6),
        "sample_size_target": SAMPLE_N,
        "run_tag": RUN_TAG
    })

    print(f"AUC={auc:.4f}  Acc={acc:.4f}  F1={f1:.4f}  TrainTime={train_time:.2f}s")



Training: LogisticRegression
AUC=1.0000  Acc=1.0000  F1=1.0000  TrainTime=62.75s

Training: DecisionTree
AUC=1.0000  Acc=1.0000  F1=1.0000  TrainTime=14.93s

Training: RandomForest
AUC=1.0000  Acc=1.0000  F1=1.0000  TrainTime=56.34s

Training: GBT
AUC=1.0000  Acc=1.0000  F1=1.0000  TrainTime=47.30s


In [14]:
# 14) Save metrics as CSV in METRICS directory (folder + single file)
metrics_df = spark.createDataFrame(results)
metrics_df.coalesce(1).write.mode("overwrite").option("header", True).csv(METRICS_OUT_DIR)

part_files = glob.glob(os.path.join(METRICS_OUT_DIR, "part-*.csv"))
if not part_files:
    raise RuntimeError("Could not find metrics part-*.csv in: " + METRICS_OUT_DIR)
shutil.copy(part_files[0], METRICS_SINGLE_CSV)

print("\n Metrics folder:", METRICS_OUT_DIR)
print(" Metrics single CSV:", METRICS_SINGLE_CSV)


 Metrics folder: /content/drive/MyDrive/7006SCN_Out/outputs/model_metrics/metrics_20260227_082713
 Metrics single CSV: /content/drive/MyDrive/7006SCN_Out/outputs/model_metrics/model_metrics_20260227_082713.csv


In [15]:
# 15) Cleanup
train_df.unpersist()
test_df.unpersist()

print("\nDONE")
print("Sample saved to:", SAMPLE_SINGLE_CSV)
print("Metrics saved to:", METRICS_SINGLE_CSV)


DONE
Sample saved to: /content/drive/MyDrive/7006SCN_Out/outputs/sample_data/sample_120k_20260227_082713.csv
Metrics saved to: /content/drive/MyDrive/7006SCN_Out/outputs/model_metrics/model_metrics_20260227_082713.csv
